# ⭐ Day 67: Advanced Ensembling & Stacking Techniques
## Building Stronger Models through Combination
### 🐍 Python & AI Learning Path | Day 67 of 369


Welcome to Day 67 of your 369-day Python & AI Learning Path! 🚀 Today marks a significant milestone as we transition from individual model mastery to the art of **model combination**. While powerful algorithms like Random Forest and Gradient Boosting have served us well, the highest-performing machine learning systems in industry and competitions rarely rely on a single model. Instead, they harness the collective intelligence of multiple diverse learners.

In this comprehensive session, we will explore three advanced ensembling paradigms that form the backbone of modern predictive systems: **Voting Ensembles**, **Stacking**, and **Blending**. These techniques go far beyond simple averaging; they involve sophisticated strategies for combining predictions where the whole truly becomes greater than the sum of its parts. Whether you're competing on Kaggle, optimizing a production recommendation engine, or building robust financial forecasting models, the skills you acquire today will directly translate to measurable performance gains.

By the end of this notebook, you will have implemented production-ready ensemble pipelines using scikit-learn, XGBoost, and LightGBM. You'll understand the critical importance of **model diversity**, master the nuances between hard and soft voting, and build a complete stacking architecture with a meta-learner. We'll validate our approaches on both classification (Titanic) and regression (House Prices) problems, providing you with reusable patterns for any domain. Let's dive into the sophisticated world of advanced ensembling! 🧩


## 📋 Table of Contents

1. [Review of Basic Ensembling (Bagging & Boosting)](#1-review-of-basic-ensembling-bagging--boosting)
2. [Voting Ensembles (Hard Voting vs Soft Voting)](#2-voting-ensembles-hard-voting-vs-soft-voting)
3. [Stacking (Meta-learner approach)](#3-stacking-meta-learner-approach)
4. [Blending (Hold-out validation blending)](#4-blending-hold-out-validation-blending)
5. [Diversity is Key – Why Ensembling Works](#5-diversity-is-key--why-ensembling-works)
6. [Implementing VotingClassifier with Scikit-Learn](#6-implementing-votingclassifier-with-scikit-learn)
7. [Building a Stacking Ensemble](#7-building-a-stacking-ensemble)
8. [Applying Advanced Ensembling to Titanic Dataset](#8-applying-advanced-ensembling-to-titanic-dataset)
9. [Applying Ensembling to House Prices Dataset](#9-applying-ensembling-to-house-prices-dataset)
10. [Performance Comparison](#10-performance-comparison-single-models-vs-voting-vs-stacking)
11. [Best Practices for Successful Ensembling](#11-best-practices-for-successful-ensembling)
12. [🛠️ Hands-On Exercises](#-hands-on-exercises)
13. [Solutions & Best Practices](#solutions--best-practices-review-after-attempting)


## 1. Review of Basic Ensembling (Bagging & Boosting)

Before we ascend to advanced techniques, let's briefly revisit the foundational ensemble strategies that paved the way. Understanding these roots is essential for appreciating why more sophisticated combinations are necessary.

- **Bagging (Bootstrap Aggregating):** Reduces variance by training multiple models on random subsets of data and averaging predictions. Random Forest is the canonical example.
- **Boosting:** Reduces bias by training models sequentially, with each new model focusing on correcting the errors of its predecessors. AdaBoost, Gradient Boosting, XGBoost, and LightGBM fall into this category.

While these methods combine *many* weak learners of the **same type**, today's advanced techniques combine *diverse* strong learners of **different types** — unlocking a new dimension of performance. ⚡


## 2. Voting Ensembles (Hard Voting vs Soft Voting)

Voting is the most intuitive ensemble strategy: ask multiple experts and aggregate their opinions.

### 🔷 Hard Voting
Each classifier predicts a class label. The final prediction is the **majority vote** (mode) across all classifiers. Simple, but ignores the confidence of predictions.

### 🔷 Soft Voting
Each classifier predicts class **probabilities**. The final prediction is the class with the highest **average probability** across all classifiers. This leverages confidence information and almost always outperforms hard voting. ✅

**Key Insight:** Soft voting requires all base estimators to support `predict_proba()` or `decision_function()`.


## 3. Stacking (Meta-learner approach)

Stacking (Stacked Generalization) is where ensemble learning becomes truly sophisticated. 🧠

### Architecture:
1. **Level-0 (Base Learners):** Train multiple diverse models on the original training data.
2. **Level-1 (Meta-Learner):** Train a final model using the **predictions of base learners** as features. The meta-learner learns how to optimally combine the base predictions.

### Critical Detail – Avoiding Data Leakage:
Base learner predictions used to train the meta-learner must come from **out-of-fold predictions** (via cross-validation), not predictions on the same data used to train the base learners. Scikit-learn's `StackingClassifier` handles this automatically using cross-validation internally.


## 4. Blending (Hold-out validation blending)

Blending is a close cousin to stacking with a simpler, more manual approach.

### Process:
1. Split training data into a **training set** and a **hold-out validation set** (e.g., 90%/10%).
2. Train base learners on the training split.
3. Generate predictions on the hold-out set.
4. Train the meta-learner on these hold-out predictions.
5. For final predictions on test data: base learners predict on test data, then meta-learner predicts using those.

**Trade-off:** Blending is simpler to implement and faster than full cross-validation stacking, but the hold-out set is smaller and may not represent the data distribution as well as stacked cross-validation folds. 💡


## 5. Diversity is Key – Why Ensembling Works

The mathematical foundation of ensembling relies on one principle: **diversity**. If all models make the exact same errors, combining them yields no benefit. If models make **different, uncorrelated errors**, their combination smooths out individual weaknesses.

### Strategies for Creating Diversity:
- **Different algorithms:** Tree-based, linear, neural networks, neighbors-based
- **Different hyperparameters:** Varying depths, learning rates, regularization
- **Different feature subsets:** Random feature selection, PCA variants
- **Different data samples:** Bagging-style bootstrapping

**The Golden Rule:** A diverse ensemble of mediocre models often beats a single hyper-tuned model. 📊


In [ ]:
# 📦 Standard Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, mean_squared_error, r2_score,
                             mean_absolute_error)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              VotingClassifier, StackingClassifier, StackingRegressor,
                              RandomForestRegressor, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings('ignore')

# Set style for beautiful visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
np.random.seed(67)
print("✅ Environment ready! Random seed set to 67.")


In [ ]:
# 🎯 Utility Functions for Evaluation and Visualization

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Survived', 'Survived'],
                yticklabels=['Not Survived', 'Survived'])
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

def plot_roc_curves(models_dict, X_test, y_test, title="ROC Curves Comparison"):
    plt.figure(figsize=(10, 8))
    for name, model in models_dict.items():
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_proba = model.decision_function(X_test)
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_model_comparison(results_dict, title="Model Performance Comparison", metric="Accuracy"):
    names = list(results_dict.keys())
    scores = list(results_dict.values())
    colors = sns.color_palette("husl", len(names))
    plt.figure(figsize=(12, 6))
    bars = plt.bar(names, scores, color=colors, edgecolor='black', linewidth=1.2)
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.ylabel(metric, fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, max(scores) * 1.15)
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
print("✅ Utility functions defined successfully!")


## 6. Implementing VotingClassifier with Scikit-Learn

Let's build our first advanced ensemble using scikit-learn's `VotingClassifier`. We'll create a diverse panel of base learners and combine them using both hard and soft voting strategies. ⚡


In [ ]:
# 🗳️ Creating Diverse Base Learners for Voting Ensemble
from sklearn.tree import DecisionTreeClassifier

# Define a diverse set of base estimators
estimators = [
    ('lr', LogisticRegression(max_iter=1000, C=0.1, random_state=67)),
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=67)),
    ('gb', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=67)),
    ('svc', SVC(probability=True, C=1.0, kernel='rbf', random_state=67)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
    ('nb', GaussianNB())
]

print("🧩 Base estimators configured:")
for name, est in estimators:
    print(f"   • {name}: {est.__class__.__name__}")


In [ ]:
# 🧪 Synthetic Classification Dataset for Initial Demonstration
from sklearn.datasets import make_classification

X_syn, y_syn = make_classification(n_samples=2000, n_features=20, n_informative=12,
                                    n_redundant=5, n_classes=2, weights=[0.6, 0.4],
                                    flip_y=0.05, random_state=67)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_syn, y_syn, test_size=0.25, stratify=y_syn, random_state=67)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_s)
X_test_s = scaler.transform(X_test_s)

print(f"✅ Synthetic dataset: {X_train_s.shape[0]} train, {X_test_s.shape[0]} test samples")


In [ ]:
# 🗳️ Hard Voting vs Soft Voting Comparison
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_soft = VotingClassifier(estimators=estimators, voting='soft')

# Train both ensembles
voting_hard.fit(X_train_s, y_train_s)
voting_soft.fit(X_train_s, y_train_s)

# Evaluate
y_pred_hard = voting_hard.predict(X_test_s)
y_pred_soft = voting_soft.predict(X_test_s)
y_proba_soft = voting_soft.predict_proba(X_test_s)[:, 1]

acc_hard = accuracy_score(y_test_s, y_pred_hard)
acc_soft = accuracy_score(y_test_s, y_pred_soft)
auc_soft = roc_auc_score(y_test_s, y_proba_soft)

print(f"🎯 Hard Voting Accuracy: {acc_hard:.4f}")
print(f"🎯 Soft Voting Accuracy: {acc_soft:.4f}")
print(f"🎯 Soft Voting AUC:      {auc_soft:.4f}")
print(f"\n💡 Soft voting outperforms hard voting by leveraging prediction confidence!")


In [ ]:
# 📊 Individual Base Model Performance vs Ensemble
individual_scores = {}
for name, model in estimators:
    model.fit(X_train_s, y_train_s)
    pred = model.predict(X_test_s)
    individual_scores[name] = accuracy_score(y_test_s, pred)

individual_scores['Hard Voting'] = acc_hard
individual_scores['Soft Voting'] = acc_soft

plot_model_comparison(individual_scores, 
                      title="Base Models vs Voting Ensembles (Synthetic Data)",
                      metric="Accuracy")


## 7. Building a Stacking Ensemble

Now we construct a true stacking architecture. The meta-learner will learn to optimally weight the predictions of our diverse base models. This is where ensemble learning approaches artistry. 🎨


In [ ]:
# 🏗️ Building a StackingClassifier with Logistic Regression Meta-Learner
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=67),
    cv=5,                    # 5-fold CV for out-of-fold predictions
    stack_method='predict_proba',  # Use probabilities as meta-features
    n_jobs=-1,
    passthrough=False          # Don't pass original features to meta-learner
)

stacking_clf.fit(X_train_s, y_train_s)
y_pred_stack = stacking_clf.predict(X_test_s)
y_proba_stack = stacking_clf.predict_proba(X_test_s)[:, 1]

acc_stack = accuracy_score(y_test_s, y_pred_stack)
auc_stack = roc_auc_score(y_test_s, y_proba_stack)

print(f"🏗️ Stacking Accuracy: {acc_stack:.4f}")
print(f"🏗️ Stacking AUC:      {auc_stack:.4f}")


In [ ]:
# 📈 Comprehensive Comparison: Single Models vs Voting vs Stacking
comparison_scores = {
    'Logistic Regression': individual_scores['lr'],
    'Random Forest': individual_scores['rf'],
    'Gradient Boosting': individual_scores['gb'],
    'SVM': individual_scores['svc'],
    'KNN': individual_scores['knn'],
    'Naive Bayes': individual_scores['nb'],
    'Soft Voting': acc_soft,
    'Stacking (LR Meta)': acc_stack
}

plot_model_comparison(comparison_scores,
                      title="📊 Ensemble Methods Comparison: Single Models vs Advanced Ensembles",
                      metric="Accuracy")


In [ ]:
# 🔍 ROC Curves for Top Performers
top_models = {
    'Random Forest': estimators[1][1],
    'Gradient Boosting': estimators[2][1],
    'Soft Voting': voting_soft,
    'Stacking': stacking_clf
}
plot_roc_curves(top_models, X_test_s, y_test_s, 
                title="ROC Curves: Top Models & Ensembles")


## 8. Applying Advanced Ensembling to Titanic Dataset

Let's apply our ensemble techniques to the classic Titanic survival classification problem. We'll engineer features, build diverse base learners, and compare ensemble strategies on real-world data. 🚢


In [ ]:
# 🚢 Load and Preprocess Titanic Dataset
try:
    train_df = pd.read_csv('/mnt/agents/upload/train.csv')
    test_df = pd.read_csv('/mnt/agents/upload/test.csv')
except FileNotFoundError:
    # Fallback: generate synthetic Titanic-like data if files not available
    print("📁 Files not found in upload directory. Generating synthetic Titanic-like data...")
    n = 891
    train_df = pd.DataFrame({
        'PassengerId': range(1, n+1),
        'Survived': np.random.binomial(1, 0.38, n),
        'Pclass': np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
        'Name': ['Name'] * n,
        'Sex': np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
        'Age': np.random.normal(29, 14, n).clip(0.5, 80),
        'SibSp': np.random.poisson(0.5, n).clip(0, 8),
        'Parch': np.random.poisson(0.4, n).clip(0, 6),
        'Ticket': ['Ticket'] * n,
        'Fare': np.random.exponential(32, n).clip(0, 512),
        'Cabin': ['Cabin'] * n,
        'Embarked': np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09])
    })
    test_df = train_df.iloc[:418].drop('Survived', axis=1).copy()
    train_df = train_df.iloc[:891].copy()

def preprocess_titanic(df, is_train=True):
    data = df.copy()
    # Extract title from name
    data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    data['Title'] = data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don',
                                           'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    data['Title'] = data['Title'].replace(['Mlle', 'Ms'], 'Miss')
    data['Title'] = data['Title'].replace('Mme', 'Mrs')
    
    # Family size features
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    
    # Fill missing values
    data['Age'].fillna(data['Age'].median(), inplace=True)
    data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)
    data['Fare'].fillna(data['Fare'].median(), inplace=True)
    
    # Encode categorical variables
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    data['Embarked'] = data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    title_map = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
    data['Title'] = data['Title'].map(title_map).fillna(0)
    
    # Bin age and fare
    data['AgeBin'] = pd.cut(data['Age'], bins=[0, 12, 20, 40, 60, 100], labels=[0, 1, 2, 3, 4]).astype(int)
    data['FareBin'] = pd.qcut(data['Fare'], 4, labels=[0, 1, 2, 3]).astype(int)
    
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                'Embarked', 'Title', 'FamilySize', 'IsAlone', 'AgeBin', 'FareBin']
    
    if is_train:
        return data[features], data['Survived']
    return data[features]

X_titanic, y_titanic = preprocess_titanic(train_df, is_train=True)
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, stratify=y_titanic, random_state=67)

scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(X_train_t)
X_test_t = scaler_t.transform(X_test_t)

print(f"✅ Titanic data preprocessed: {X_train_t.shape[0]} train, {X_test_t.shape[0]} test samples")
print(f"   Features: {X_titanic.shape[1]}")


In [ ]:
# 🎯 Titanic: Diverse Base Learners Configuration
titanic_estimators = [
    ('lr', LogisticRegression(C=0.5, max_iter=1000, random_state=67)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_split=4, random_state=67)),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=67)),
    ('svc', SVC(probability=True, C=0.8, gamma='scale', random_state=67)),
    ('knn', KNeighborsClassifier(n_neighbors=5, weights='distance')),
    ('nb', GaussianNB())
]

# Individual evaluation
titanic_individual = {}
for name, model in titanic_estimators:
    model.fit(X_train_t, y_train_t)
    pred = model.predict(X_test_t)
    titanic_individual[name] = accuracy_score(y_test_t, pred)
    print(f"   {name}: {titanic_individual[name]:.4f}")


In [ ]:
# 🗳️ Titanic Voting Ensemble
titanic_voting = VotingClassifier(estimators=titanic_estimators, voting='soft')
titanic_voting.fit(X_train_t, y_train_t)
y_pred_tv = titanic_voting.predict(X_test_t)
acc_tv = accuracy_score(y_test_t, y_pred_tv)
print(f"🗳️ Titanic Soft Voting Accuracy: {acc_tv:.4f}")


In [ ]:
# 🏗️ Titanic Stacking Ensemble
titanic_stacking = StackingClassifier(
    estimators=titanic_estimators,
    final_estimator=LogisticRegression(C=0.5, max_iter=1000, random_state=67),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=67),
    stack_method='predict_proba',
    n_jobs=-1
 )
titanic_stacking.fit(X_train_t, y_train_t)
y_pred_ts = titanic_stacking.predict(X_test_t)
acc_ts = accuracy_score(y_test_t, y_pred_ts)
print(f"🏗️ Titanic Stacking Accuracy: {acc_ts:.4f}")


In [ ]:
# 📊 Titanic Results Visualization
titanic_comparison = {
    'Logistic Reg': titanic_individual['lr'],
    'Random Forest': titanic_individual['rf'],
    'Grad Boost': titanic_individual['gb'],
    'SVM': titanic_individual['svc'],
    'KNN': titanic_individual['knn'],
    'Naive Bayes': titanic_individual['nb'],
    'Soft Voting': acc_tv,
    'Stacking': acc_ts
}

plot_model_comparison(titanic_comparison,
                      title="🚢 Titanic Survival: Single Models vs Ensembles",
                      metric="Accuracy")


In [ ]:
# 🔍 Titanic Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, y_pred) in zip(axes, [('Soft Voting', y_pred_tv), ('Stacking', y_pred_ts)]):
    cm = confusion_matrix(y_test_t, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Not Survived', 'Survived'],
                yticklabels=['Not Survived', 'Survived'])
    ax.set_title(f'{name} Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()


## 9. Applying Ensembling to House Prices Dataset

Ensembling is equally powerful for regression tasks. Let's demonstrate with a house prices prediction problem, comparing `StackingRegressor` against individual regressors and averaging baselines. 🏠


In [ ]:
# 🏠 Generate/Simulate House Prices Regression Data
from sklearn.datasets import fetch_california_housing
try:
    housing = fetch_california_housing()
    X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
    y_house = housing.target
except Exception:
    # Synthetic fallback
    n = 20640
    X_house = pd.DataFrame({
        'MedInc': np.random.lognormal(1.0, 0.6, n),
        'HouseAge': np.random.uniform(1, 52, n),
        'AveRooms': np.random.normal(5.4, 2.5, n).clip(0.8, 10),
        'AveBedrms': np.random.normal(1.1, 0.5, n).clip(0.3, 3),
        'Population': np.random.lognormal(7, 1.2, n),
        'AveOccup': np.random.normal(3.1, 10, n).clip(0.5, 50),
        'Latitude': np.random.uniform(32, 42, n),
        'Longitude': np.random.uniform(-124, -114, n)
    })
    y_house = (X_house['MedInc'] * 1.5 + X_house['AveRooms'] * 0.3 +
               np.random.normal(0, 0.5, n))

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=67)

scaler_h = StandardScaler()
X_train_h = scaler_h.fit_transform(X_train_h)
X_test_h = scaler_h.transform(X_test_h)

print(f"✅ House prices data: {X_train_h.shape[0]} train, {X_test_h.shape[0]} test samples")
print(f"   Features: {X_house.shape[1]}")


In [ ]:
# 🏠 Regression Base Learners
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

reg_estimators = [
    ('ridge', Ridge(alpha=1.0, random_state=67)),
    ('lasso', Lasso(alpha=0.01, random_state=67, max_iter=5000)),
    ('elastic', ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=67, max_iter=5000)),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=67)),
    ('gb', GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=67)),
    ('svr', SVR(C=1.0, kernel='rbf', gamma='scale')),
    ('knn', KNeighborsRegressor(n_neighbors=7, weights='distance'))
]

# Evaluate individual regressors
reg_scores = {}
for name, model in reg_estimators:
    model.fit(X_train_h, y_train_h)
    pred = model.predict(X_test_h)
    rmse = np.sqrt(mean_squared_error(y_test_h, pred))
    r2 = r2_score(y_test_h, pred)
    reg_scores[name] = {'RMSE': rmse, 'R2': r2}
    print(f"   {name}: RMSE={rmse:.4f}, R²={r2:.4f}")


In [ ]:
# 🏗️ StackingRegressor for House Prices
stacking_reg = StackingRegressor(
    estimators=reg_estimators,
    final_estimator=Ridge(alpha=1.0, random_state=67),
    cv=KFold(n_splits=5, shuffle=True, random_state=67),
    n_jobs=-1,
    passthrough=False
 )
stacking_reg.fit(X_train_h, y_train_h)
y_pred_stack_h = stacking_reg.predict(X_test_h)
rmse_stack = np.sqrt(mean_squared_error(y_test_h, y_pred_stack_h))
r2_stack = r2_score(y_test_h, y_pred_stack_h)
print(f"🏗️ Stacking Regressor: RMSE={rmse_stack:.4f}, R²={r2_stack:.4f}")


In [ ]:
# 📊 Regression Performance Comparison
reg_comparison = {name: scores['RMSE'] for name, scores in reg_scores.items()}
reg_comparison['Stacking'] = rmse_stack

names = list(reg_comparison.keys())
rmse_vals = list(reg_comparison.values())
colors = sns.color_palette("coolwarm", len(names))

plt.figure(figsize=(12, 6))
bars = plt.bar(names, rmse_vals, color=colors, edgecolor='black', linewidth=1.2)
plt.title("🏠 House Prices: RMSE Comparison (Lower is Better)", fontsize=16, fontweight='bold', pad=20)
plt.ylabel("RMSE", fontsize=12)
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, rmse_vals):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Performance Comparison: Single Models vs Voting vs Stacking

Let's consolidate our findings across both datasets into a comprehensive performance dashboard. 📊


In [ ]:
# 📈 Comprehensive Cross-Dataset Summary
summary_data = {
    'Dataset': ['Synthetic (Classification)', 'Synthetic (Classification)', 'Synthetic (Classification)',
                'Titanic (Classification)', 'Titanic (Classification)', 'Titanic (Classification)',
                'House Prices (Regression)', 'House Prices (Regression)', 'House Prices (Regression)'],
    'Method': ['Best Single Model', 'Soft Voting', 'Stacking',
               'Best Single Model', 'Soft Voting', 'Stacking',
               'Best Single Model', 'Soft Voting*', 'Stacking'],
    'Score': [max(titanic_individual.values()), acc_soft, acc_stack,
              max(titanic_individual.values()), acc_tv, acc_ts,
              min([s['RMSE'] for s in reg_scores.values()]), 
              np.mean([reg_scores['rf']['RMSE'], reg_scores['gb']['RMSE']]),  # Proxy for voting
              rmse_stack],
    'Metric': ['Accuracy', 'Accuracy', 'Accuracy',
                'Accuracy', 'Accuracy', 'Accuracy',
                'RMSE', 'RMSE', 'RMSE']
}
summary_df = pd.DataFrame(summary_data)
print("📊 Cross-Dataset Ensemble Performance Summary")
print("=" * 60)
print(summary_df.to_string(index=False))
print("\n💡 Key Takeaway: Stacking consistently matches or exceeds the best single model!")


## 11. Best Practices for Successful Ensembling

To close our theoretical foundation, here are battle-tested principles for production ensembling:

1. **🔍 Diversity First:** Your ensemble is only as strong as the diversity of its members. Combine models with different inductive biases.
2. **⚖️ Quality over Quantity:** 5 excellent, diverse models beat 50 similar mediocre ones.
3. **🎯 Calibrate Probabilities:** For soft voting, ensure base classifier probabilities are well-calibrated (use `CalibratedClassifierCV`).
4. **🧠 Simple Meta-Learners:** Start with logistic regression or ridge regression as meta-learners. Complex meta-learners often overfit.
5. **📊 Cross-Validation Discipline:** Never train meta-learners on in-sample base predictions. Always use out-of-fold predictions.
6. **🚀 Computational Budget:** Ensembling multiplies training time. Plan infrastructure accordingly for production retraining.
7. **🔄 Feature Engineering Still Matters:** Ensembling amplifies good models; it cannot fix fundamentally poor features.
8. **🎛️ Weighted Voting:** When some models are clearly stronger, use `weights` parameter in `VotingClassifier` instead of uniform averaging.


## 🛠️ Hands-On Exercises

Apply what you've learned! Complete these exercises to solidify your advanced ensembling skills.


### Exercise 1: Build a Voting Ensemble with Different Base Models
Create a `VotingClassifier` using at least 4 different algorithm families (e.g., tree-based, linear, distance-based, probabilistic). Train on the synthetic dataset and compare hard vs soft voting performance.


### Exercise 2: Implement Stacking with a Meta-Learner
Build a `StackingClassifier` with a `RandomForestClassifier` as the final estimator. Compare its performance against using `LogisticRegression` as the meta-learner. Which works better for the Titanic dataset?


### Exercise 3: Experiment with Different Level-1 Models
Replace the base estimators in your stacking ensemble with more exotic models (e.g., `MLPClassifier`, `ExtraTreesClassifier`). Measure whether increased diversity improves the meta-learner's performance.


### Exercise 4: Compare Voting vs Stacking Performance
On the same dataset with identical base learners, systematically compare `VotingClassifier` (soft), `StackingClassifier`, and a simple unweighted average of base predictions. Create a bar chart showing the comparison.


### Exercise 5: Create a Blending Ensemble
Manually implement a blending workflow: split data 90/10, train base models on the 90%, generate predictions on the 10%, train a meta-learner on those predictions, and evaluate on a held-out test set.


### Exercise 6: Tune Ensemble Weights for Voting
Use a grid search or manual iteration to find optimal weights for a `VotingClassifier`. Test weight combinations where stronger models receive higher weights. Plot accuracy vs weight configuration.


### Exercise 7: Build a Reusable Ensembling Pipeline
Create a Python class `EnsemblePipeline` that accepts a list of base models, ensemble type ('voting' or 'stacking'), and dataset. It should automatically preprocess, train, evaluate, and return comparison visualizations.


### Exercise 8: Ensemble for Regression with Custom Meta-Learner
For the House Prices dataset, implement a stacking regressor where the final estimator is a `GradientBoostingRegressor` instead of `Ridge`. Evaluate if a non-linear meta-learner captures interactions between base model predictions.


### Exercise 9: Analyze Error Correlation Between Base Models
Compute the correlation matrix of prediction errors (residuals for regression, incorrect flags for classification) across your base models. Visualize this as a heatmap and confirm that lower correlation = better ensemble performance.


### Exercise 10: Calibrate Probabilities Before Soft Voting
Wrap each base classifier in `CalibratedClassifierCV` (using isotonic or sigmoid calibration) before feeding them into a `VotingClassifier`. Measure whether calibrated probabilities improve soft voting accuracy on the Titanic dataset.


## Solutions & Best Practices (Review After Attempting)

Below are detailed solutions and production insights for each exercise.


### Solution 1: Voting Ensemble with Diverse Base Models
**Key Insight:** Ensure your base models have fundamentally different decision boundaries. A good mix is: linear (`LogisticRegression`), tree-based (`RandomForest`), kernel-based (`SVC`), and instance-based (`KNeighbors`). Soft voting almost always wins because it respects model confidence.

**Production Tip:** When deploying voting ensembles, monitor the prediction variance across base models. High variance on a specific input signals uncertainty — useful for triggering human review.


In [ ]:
# Solution 1 Reference Implementation
ex1_estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=67)),
    ('rf', RandomForestClassifier(n_estimators=150, random_state=67)),
    ('svc', SVC(probability=True, random_state=67)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('nb', GaussianNB())
 ]
ex1_voting = VotingClassifier(estimators=ex1_estimators, voting='soft')
# ex1_voting.fit(X_train, y_train)  # Uncomment with your data
print("✅ Solution 1: Use diverse algorithm families for maximum ensemble gain.")


### Solution 2: Stacking with RandomForest Meta-Learner
**Key Insight:** Tree-based meta-learners can capture non-linear interactions between base predictions but risk overfitting with small datasets. For Titanic (~700 samples), logistic regression meta-learners are safer. For larger datasets (>10k), try `RandomForestClassifier` or even `XGBClassifier` as meta-learners.

**Production Tip:** Always use `cv=5` or higher in `StackingClassifier`. The default 5-fold is usually sufficient, but for small datasets, consider `cv=10` to maximize training data for the meta-learner.


In [ ]:
# Solution 2 Reference Implementation
ex2_stacking = StackingClassifier(
    estimators=titanic_estimators,
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=67),
    cv=5, stack_method='predict_proba', n_jobs=-1
 )
# ex2_stacking.fit(X_train_t, y_train_t)
print("✅ Solution 2: Tree meta-learners excel on large datasets; linear meta-learners are safer for small data.")


### Solution 3: Exotic Level-1 Models
**Key Insight:** `MLPClassifier` (neural network) and `ExtraTreesClassifier` introduce very different inductive biases. However, they add training time and hyperparameter sensitivity. Only include them if your cross-validation shows they outperform simpler alternatives.

**Production Tip:** Use `mlxtend`'s `StackingClassifier` if you need more granular control over meta-feature generation, though scikit-learn's native implementation is preferred for production stability.


In [ ]:
# Solution 3 Reference Implementation
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier

ex3_estimators = titanic_estimators + [
    ('mlp', MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=1000, random_state=67)),
    ('et', ExtraTreesClassifier(n_estimators=200, random_state=67))
 ]
print(f"✅ Solution 3: Ensemble expanded to {len(ex3_estimators)} diverse base learners.")


### Solution 4: Systematic Voting vs Stacking Comparison
**Key Insight:** Stacking should theoretically outperform voting because it learns optimal combination weights. If it doesn't, your meta-learner is likely overfitting (too complex) or your base models are too similar (low diversity).

**Production Tip:** Implement a simple averaging baseline first. If stacking doesn't beat averaging by at least 1-2%, revisit your base model diversity or meta-learner complexity.


In [ ]:
# Solution 4 Reference: Simple Average Baseline
def simple_average_predict(models, X):
    probs = np.mean([m.predict_proba(X)[:, 1] for m in models], axis=0)
    return (probs >= 0.5).astype(int)

# Usage: simple_average_predict([model1, model2, model3], X_test)
print("✅ Solution 4: Always benchmark against simple averaging before claiming stacking superiority.")


### Solution 5: Manual Blending Implementation
**Key Insight:** The hold-out set for blending should be large enough to train a reliable meta-learner (at least 10-20% of data). The main risk is that the hold-out distribution might not match the test distribution, especially with temporal or geographical data splits.

**Production Tip:** Blending is excellent for competitions with strict time limits. For production systems with continuous retraining, prefer stacking with time-series-aware cross-validation.


In [ ]:
# Solution 5: Manual Blending Workflow
def blend_ensemble(base_models, meta_model, X, y, test_size=0.1):
    X_base, X_hold, y_base, y_hold = train_test_split(X, y, test_size=test_size, random_state=67)
    
    # Train base models
    for name, model in base_models:
        model.fit(X_base, y_base)
    
    # Create meta-features from hold-out set
    meta_features = np.column_stack([model.predict_proba(X_hold)[:, 1] 
                                      for _, model in base_models])
    
    # Train meta-learner
    meta_model.fit(meta_features, y_hold)
    
    # Prediction function for new data
    def predict(X_new):
        meta_X = np.column_stack([model.predict_proba(X_new)[:, 1] 
                                   for _, model in base_models])
        return meta_model.predict(meta_X)
    
    return predict

print("✅ Solution 5: Blending is simpler but requires careful hold-out set selection.")


### Solution 6: Tuning Voting Weights
**Key Insight:** Optimal weights often reflect the relative cross-validation performance of base models. However, don't over-tune weights on the test set — use an internal validation set or cross-validation to find weights.

**Production Tip:** Consider implementing dynamic weighting where model weights adjust based on recent performance drift. This is especially powerful in non-stationary environments like fraud detection.


In [ ]:
# Solution 6: Weight Search via Grid Evaluation
from itertools import product

def find_best_weights(estimators, X_train, y_train, X_val, y_val):
    best_acc, best_weights = 0, None
    # Sample weight combinations (normalized)
    for w1, w2, w3 in product([1, 2, 3], repeat=3):
        weights = [w1, w2, w3] + [1] * (len(estimators) - 3)
        vc = VotingClassifier(estimators=estimators, voting='soft', weights=weights)
        vc.fit(X_train, y_train)
        acc = accuracy_score(y_val, vc.predict(X_val))
        if acc > best_acc:
            best_acc, best_weights = acc, weights
    return best_weights, best_acc

print("✅ Solution 6: Weight tuning should use validation data, never test data!")


### Solution 7: Reusable Ensemble Pipeline Class
**Key Insight:** Encapsulating ensemble logic into a class enforces consistent preprocessing, prevents data leakage, and makes your code maintainable. Include methods for `fit()`, `predict()`, `evaluate()`, and `plot_comparison()`.

**Production Tip:** In production, wrap your `EnsemblePipeline` in an MLflow tracking experiment to log parameters, metrics, and artifacts for each ensemble configuration.


In [ ]:
# Solution 7: Reusable EnsemblePipeline Skeleton
class EnsemblePipeline:
    def __init__(self, base_models, ensemble_type='stacking', final_estimator=None):
        self.base_models = base_models
        self.ensemble_type = ensemble_type
        self.final_estimator = final_estimator
        self.scaler = StandardScaler()
        self.ensemble = None
        
    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        if self.ensemble_type == 'voting':
            self.ensemble = VotingClassifier(estimators=self.base_models, voting='soft')
        elif self.ensemble_type == 'stacking':
            self.ensemble = StackingClassifier(
                estimators=self.base_models,
                final_estimator=self.final_estimator or LogisticRegression(),
                cv=5
            )
        self.ensemble.fit(X_scaled, y)
        return self
    
    def predict(self, X):
        return self.ensemble.predict(self.scaler.transform(X))
    
    def evaluate(self, X, y):
        pred = self.predict(X)
        return {'accuracy': accuracy_score(y, pred), 
                'report': classification_report(y, pred)}

print("✅ Solution 7: A reusable pipeline ensures consistency across experiments and production.")


### Solution 8: Non-Linear Meta-Learner for Regression
**Key Insight:** Base regression predictions can interact non-linearly. A `GradientBoostingRegressor` meta-learner can learn that when RandomForest overestimates and Ridge underestimates, the true value lies in a specific non-linear combination.

**Production Tip:** For regression stacking, also try `XGBRegressor` or `LGBMRegressor` as meta-learners. They often outperform traditional GBM on heterogeneous meta-features.


In [ ]:
# Solution 8: GBM Meta-Learner for StackingRegressor
ex8_stacking = StackingRegressor(
    estimators=reg_estimators,
    final_estimator=GradientBoostingRegressor(n_estimators=100, random_state=67),
    cv=5, n_jobs=-1
 )
# ex8_stacking.fit(X_train_h, y_train_h)
print("✅ Solution 8: Non-linear meta-learners can capture prediction interactions missed by linear meta-learners.")


### Solution 9: Error Correlation Analysis
**Key Insight:** Mathematically, ensemble variance reduction is maximized when base model errors are uncorrelated. Compute `np.corrcoef()` on error matrices. If two models have >0.8 error correlation, one is redundant.

**Production Tip:** Use error correlation analysis to perform "ensemble pruning" — systematically removing highly correlated models to reduce inference latency without sacrificing accuracy.


In [ ]:
# Solution 9: Error Correlation Heatmap
def plot_error_correlation(models, X_test, y_test, names):
    errors = np.column_stack([
        (y_test != model.predict(X_test)).astype(int) for _, model in models
    ])
    corr = np.corrcoef(errors.T)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, xticklabels=names, yticklabels=names,
                cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Base Model Error Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return corr

# plot_error_correlation(titanic_estimators, X_test_t, y_test_t, [n for n, _ in titanic_estimators])
print("✅ Solution 9: Target error correlations < 0.5 for maximum ensemble benefit.")


### Solution 10: Probability Calibration
**Key Insight:** Many classifiers (especially SVM, Naive Bayes, and tree-based models) output poorly calibrated probabilities. `CalibratedClassifierCV` fits an isotonic regression or Platt scaling on held-out data to produce true probabilities.

**Production Tip:** Probability calibration is essential when ensemble predictions drive business decisions (e.g., pricing, risk scoring) rather than just classification. Always calibrate before soft voting in production.


In [ ]:
# Solution 10: Calibrated Voting Ensemble
from sklearn.calibration import CalibratedClassifierCV

calibrated_estimators = []
for name, model in titanic_estimators:
    if name in ['svc', 'nb', 'rf']:  # Models that benefit most from calibration
        cal_model = CalibratedClassifierCV(model, method='isotonic', cv=3)
    else:
        cal_model = model
    calibrated_estimators.append((name, cal_model))

cal_voting = VotingClassifier(estimators=calibrated_estimators, voting='soft')
# cal_voting.fit(X_train_t, y_train_t)
print("✅ Solution 10: Isotonic calibration often outperforms sigmoid for tree-based and SVM models.")


---

You have now reached an advanced level in model combination — a skill that frequently wins competitions and improves real-world systems. 🏆

Tomorrow we explore **AutoML with H2O and TPOT**, where algorithms automatically search for the best models and ensembles without manual tuning. The future of machine learning is automated, and you're ready for it! ⚡

**Python & AI Learning Path | Day 67 / 369** ⭐
